# Lektion 10: Knowledge Graphs – Wissen als Beziehungen



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 01–09 (besonders Build 2 und seine Grenze) |
| **Du baust** | Deinen ersten Knowledge Graph – zuerst als reines Python-Dict, dann als echte Graph-Datenstruktur |

## 🎯 Nach dieser Lektion kannst du:

- erklären, warum Beziehungsfragen („Wer arbeitet woran?") Text-Suche sprengen

- einen Graph als Python-Dictionary darstellen (Nodes, Kanten, Properties)

- mit dem `GraphStore` Nodes, Labels und Beziehungen anlegen

- per **Traversierung** Fragen beantworten, die in keinem Text stehen (**Emergenz**)

- den Graph als Mermaid-Diagramm visualisieren

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Das Ermittlungsbrett

**WARUM:** Stell die Frage **„Wie hängt Ben mit dem AI-Assistant-Projekt zusammen?"** – kein Abschnitt der Dokumente enthält diesen Satz! Text-Suche (Build 2) findet sinngleiche **Texte**, aber folgt **keinen Verbindungen**.

**WAS:** Ein **Knowledge Graph** speichert Wissen als **Entities** (Dinge) und **Relationships** (Verbindungen). **Metapher:** das **Ermittlungsbrett** aus Krimiserien: Fotos (Nodes) an einer Pinnwand, verbunden mit roten Fäden (Relationships), Notizen an jedem Foto (Properties). Einem Faden nachgehen (**Traversierung**) enthüllt Muster, die auf keinem einzelnen Foto stehen.

**Wichtig:** Der Graph speichert Verbindungen **als Daten** – nicht als Sätze in Textform. Dadurch ist „Wer arbeitet woran?" eine *strukturelle* Frage, keine Textsuche.

## Schritt 1: Der Graph als pure Python-Datenstruktur

Zuerst – wie bei Build 1! – ganz ohne Werkzeug: ein Dictionary. Jede Person mappt auf das, womit sie verbunden ist:

In [2]:
graph_als_dict = {
    "Anna": ["Acme", "Engineering", "AI Assistant"],
    "Ben": ["Acme", "Data & Analytics"],
    "Engineering": ["AI Assistant"],
}

# Frage: Mit wem/ was ist Anna verbunden?
print("Anna verbunden mit:", graph_als_dict["Anna"])
print("Ben verbunden mit:", graph_als_dict["Ben"])

Anna verbunden mit: ['Acme', 'Engineering', 'AI Assistant']
Ben verbunden mit: ['Acme', 'Data & Analytics']


### 🔎 Was sehen wir hier – und was fehlt?
Das Dict kann already Verbindungen speichern – aber es fehlen: **Richtungen** (wer → wen?), **Typen** der Verbindung (arbeitet_für? gehört_zu?) und **Notizen** (Properties). Für echte Anwendungen brauchen wir mehr Struktur – und die kommt jetzt.

## Schritt 2: Der GraphStore (unser Trainings-Graph)

Die Akademie enthält `lernakademie.graph_store.GraphStore` – das Python-Pendant zu einer Graph-Datenbank (wie der Mock zum LLM). Konzepte, die du hier lernst, gelten 1:1 in **Neo4j** (Lektion 11).

**Vokabular:**
- **Node** = Karteikarte mit **Label** (Typ, z. B. `Person`) und **Properties** (`name`, `rolle`)
- **Relationship** = gerichteter Faden mit **Typ** (`WORKS_FOR`) und optionalen Properties (`seit: 2021`)
- **WICHTIG:** Kanten verbinden **IDs**, nie Namen – sonst werden zwei Annas zu einer!

In [3]:
from lernakademie.graph_store import GraphStore

store = GraphStore()

# Nodes anlegen (IDs merken!)
anna = store.add_node(["Person"], name="Anna", rolle="Entwicklerin")
ben = store.add_node(["Person"], name="Ben", rolle="Analyst")
acme = store.add_node(["Company"], name="Acme", branche="Technologie")
engineering = store.add_node(["Department"], name="Engineering")
projekt = store.add_node(["Project"], name="AI Assistant", status="aktiv")

# Beziehungen (gerichtet: von -> zu, Typ in GROSSBUCHSTABEN)
store.add_relationship(anna, acme, "WORKS_FOR", seit=2021)
store.add_relationship(anna, engineering, "BELONGS_TO")
store.add_relationship(engineering, projekt, "WORKS_ON")
store.add_relationship(ben, acme, "WORKS_FOR")

print(f"{len(store.nodes())} Nodes, {len(store.relationships())} Kanten")
for node in store.nodes():
    print(" ", node.properties)

5 Nodes, 4 Kanten
  {'name': 'Anna', 'rolle': 'Entwicklerin'}
  {'name': 'Ben', 'rolle': 'Analyst'}
  {'name': 'Acme', 'branche': 'Technologie'}
  {'name': 'Engineering'}
  {'name': 'AI Assistant', 'status': 'aktiv'}


## Schritt 3: Traversierung – den Fäden folgen

Jetzt die Superkraft des Graphen: Fragen durch **Nachbarn folgen** beantworten.

In [4]:
anna_node = store.find_node("Person", name="Anna")
print("Annas direkte Nachbarn:")
for kante, ziel in store.neighbors(anna_node.id):
    print(f"  Anna --{kante.typ}--> {ziel.properties['name']}")

Annas direkte Nachbarn:
  Anna --WORKS_FOR--> Acme
  Anna --BELONGS_TO--> Engineering


## ✏️ Deine Aufgabe: Emergenz beweisen – zweistufige Traversierung

Du beantwortest eine Frage, die **in keinem Text steht** – nur durch Kombination von zwei Fäden.

**Anforderungen:**

- Schreibe `verbunden_ueber_zwei_schritte(store, start_name, ziel_name)`, das True/False zurückgibt, ob zwei Entities über maximal zwei Kanten verbunden sind (Nutze `neighbors` zweimal – kein `path`!).

- Wende es an: Ist **Ben** über zwei Schritte mit **AI Assistant** verbunden?

- Zusatzausgabe: Gib den gefundenen Weg lesbar aus (z. B. `Ben --WORKS_FOR--> Acme`…). Tipp: Ben → Acme ist ein Sackgasse – der Weg läuft über Data & Analytics, wenn du die Kante ergänzt! Füge dafür zunächst `BELONGS_TO` für Ben und `PROVIDES_DATA_FOR` für Data & Analytics hinzu (siehe Lösung).

**Schreibe deinen Code in die folgende Zelle:**

In [5]:
daten_node = store.find_node("Department", name="Data & Analytics")
if daten_node is None:
    daten_analytik_id = store.add_node(["Department"], name="Data & Analytics")
else:
    daten_analytik_id = daten_node.id

ben_node = store.find_node("Person", name="Ben")
projekt_node = store.find_node("Project", name="AI Assistant")
store.add_relationship(ben_node.id, daten_analytik_id, "BELONGS_TO")
store.add_relationship(daten_analytik_id, projekt_node.id, "PROVIDES_DATA_FOR")

def verbunden_ueber_zwei_schritte(store, start_name, ziel_name):
    start = store.find_node(None, name=start_name)
    if start is None:
        return False
    for kante1, nachbar in store.neighbors(start.id):
        if nachbar.properties.get("name") == ziel_name:
            return True
        for kante2, ziel2 in store.neighbors(nachbar.id):
            if ziel2.properties.get("name") == ziel_name:
                print(f"  Weg: {start_name} --{kante1.typ}--> "
                      f"{nachbar.properties['name']} --{kante2.typ}--> {ziel_name}")
                return True
    return False

print(verbunden_ueber_zwei_schritte(store, "Ben", "AI Assistant"))

  Weg: Ben --BELONGS_TO--> Data & Analytics --PROVIDES_DATA_FOR--> AI Assistant
True


**💡 Warum diese Loesung funktioniert:**

Der Weg `Ben --BELONGS_TO--> Data & Analytics --PROVIDES_DATA_FOR--> AI Assistant` steht **nirgends** in den Dokumenten als Satz – er entsteht durch das **Kombinieren** zweier Fakten. Das ist **Emergenz**: Wissen, das größer ist als die Summe seiner Teile. Genau hier liegt der Grund, warum Build 3 einen Knowledge Graph braucht.

### ✅ Richtig, wenn:

- [ ] Ben ist über zwei Schritte mit AI Assistant verbunden (True) – über Data & Analytics.

- [ ] Die Funktion nutzt zwei verschachtelte `neighbors`-Aufrufe (kein `path`).

- [ ] Der Weg wird lesbar ausgegeben.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [6]:
assert verbunden_ueber_zwei_schritte(store, "Ben", "AI Assistant") is True,     "Ben sollte (ueber Data & Analytics) mit dem Projekt verbunden sein!"
assert verbunden_ueber_zwei_schritte(store, "Anna", "Acme") is True
print("Alle Prüfungen bestanden! ✅ Der Weg existiert – obwohl er in keinem Text steht.")

  Weg: Ben --BELONGS_TO--> Data & Analytics --PROVIDES_DATA_FOR--> AI Assistant
Alle Prüfungen bestanden! ✅ Der Weg existiert – obwohl er in keinem Text steht.


## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Formuliere eine Frage aus deinem Leben, die ein Graph leicht, ein Textarchiv schwer beantwortet.

2. Was würde passieren, wenn du Beziehungen als Property modellieren würdest (`arbeite_bei: "Acme"`) statt als Kante?

## 🧩 Challenge (freiwillig)

Erzeuge mit `store.to_mermaid()` den Diagramm-Code und speichere ihn in eine Datei `mein_graph.md` (mit ```mermaid-Block) – GitHub rendert ihn!

In [7]:
# YOUR CODE HERE (Challenge)

## ➡️ Weiter

**Weiter mit Lektion 11:** `notebooks/06_knowledge_graph/11_neo4j_und_cypher.ipynb` – dieselben Konzepte in einer **echten Graph-Datenbank** mit der Abfragesprache **Cypher**.